# Module 03 — AI Agents
## Lesson 9 — Deterministic Workflows vs Agents

**Core question:** should this problem be an agent at all?

Use the minimum autonomy that solves the problem well. LLM usage and agentic orchestration are separate design choices.


## Three architecture shapes

- **Workflow:** application code owns sequence and branching.
- **Agent:** model participates in choosing the next action based on observations.
- **Hybrid:** deterministic orchestration surrounds a bounded agentic region.

An LLM inside a fixed pipeline is still a workflow if the model is not choosing control flow.


## Decision questions

Ask whether the sequence is known, whether branches are a small finite rule set, whether semantic judgment is needed, whether actions are consequential, and whether adaptive reasoning materially improves results enough to justify cost and nondeterminism.


## Main example — hotel selection

Goal: find a suitable Melbourne hotel under A$300, prefer refundable options, and book only after approval.

First implement the part that can be expressed cleanly as deterministic policy.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Hotel:
    name: str
    total_aud: int
    refundable: bool
    rating: float

HOTELS = [
    Hotel("Central Stay", 260, True, 4.2),
    Hotel("River Rooms", 220, False, 4.6),
    Hotel("Garden Hotel", 290, True, 4.5),
    Hotel("Premium Suites", 360, True, 4.8),
]

def choose_hotel_deterministically(hotels: list[Hotel], max_price: int) -> Hotel:
    eligible = [hotel for hotel in hotels if hotel.total_aud <= max_price]
    if not eligible:
        raise ValueError("No hotel fits the budget")
    return max(eligible, key=lambda h: (h.refundable, h.rating, -h.total_aud))

choice = choose_hotel_deterministically(HOTELS, max_price=300)
choice


## Why this does not need an agent

The hard budget and ranking policy are explicit. Asking a model to choose the same sequence and rediscover the same preference ordering on every run would add latency and nondeterminism without clear value.


## Where agentic behavior may help

Now add ambiguous preferences: “quiet but central, good for a child, close to evening activities, avoid nightlife-heavy streets.” The model may need to inspect neighbourhood context or cancellation detail depending on what it observes. That adaptive evidence gathering is a genuine agentic region.


## Hybrid booking architecture

1. Deterministically validate dates, travellers, hard budget, identity, and policy.
2. Use bounded agentic research/comparison for ambiguous preferences.
3. Deterministically re-check hard constraints.
4. Require human approval for the exact booking.
5. Execute through an idempotent booking API and audit the result.

Autonomy is concentrated where semantic judgment adds value; authority remains outside the model.


## Represent the architecture discussion explicitly

The following toy classifier is an engineering discussion aid, not a runtime model requirement.


In [ ]:
from enum import Enum

class Architecture(str, Enum):
    WORKFLOW = "workflow"
    AGENT = "agent"
    HYBRID = "hybrid"

@dataclass(frozen=True)
class TaskCharacteristics:
    sequence_known: bool
    open_ended_decisions: bool
    consequential_side_effects: bool
    semantic_judgment_needed: bool

def recommend_architecture(task: TaskCharacteristics) -> Architecture:
    if task.open_ended_decisions and task.consequential_side_effects:
        return Architecture.HYBRID
    if task.open_ended_decisions or (task.semantic_judgment_needed and not task.sequence_known):
        return Architecture.AGENT
    return Architecture.WORKFLOW

examples = {
    "password_reset": TaskCharacteristics(True, False, True, False),
    "incident_investigation": TaskCharacteristics(False, True, False, True),
    "travel_booking": TaskCharacteristics(False, True, True, True),
}

for name, task in examples.items():
    print(name, "→", recommend_architecture(task).value)


## LLM does not imply agent

This remains a deterministic workflow:

```python
document = load_pdf()
fields = llm_extract_fields(document)
validated = validate(fields)
save(validated)
```

The LLM performs a semantic operation, but application code still determines the sequence.


## Production pattern: deterministic shell, agentic island

Keep authentication, validation, budgets, persistence, approvals, and side effects deterministic. Give the model bounded autonomy for exploration, semantic comparison, or adaptive investigation. Use structured boundaries so the agentic region can be observed, evaluated, and replaced if needed.


## Exercises

1. Classify password reset, support-ticket routing, incident investigation, invoice ingestion, and itinerary planning.
2. Change the hotel ranking policy and test the deterministic result.
3. Add “good for a quiet family evening” and identify the exact evidence-gathering decision that benefits from an agent.
4. Draw a hybrid Kubernetes incident assistant that investigates autonomously but requires approval before mutation.
5. Take one real system and mark every model-controlled branch; remove autonomy where it adds no clear value.

Next: **Lesson 10 — Build a Real Agent from Scratch in Python**.
